# Notebook de KNN

Este es el notebook 01 del proyecto `Recommend Supplier`.

- En este notebook construimos el baseline con `KNN` (`KNeighborsClassifier`) para tener una primera referencia de rendimiento.
- `KNN` clasifica por similitud entre observaciones; en esta versión trabajamos con distancia euclídea (Minkowski `p=2`) tras el preprocesado correspondiente.
- El objetivo aquí no es cerrar el modelo final, sino evaluar si `KNN` aporta señal frente a baselines en validación temporal.
- El proceso de ETL se desarrolló con asistencia de IA por el volumen y heterogeneidad de fuentes, y está documentado para su trazabilidad en `docs/privacy_checklist.md` y en `src/prompts/`.

Por privacidad, el repositorio público no incluye `raw data` ni piezas sensibles de extracción.

## Librerías

In [ ]:
# Librerías

import pandas as pd
pd.set_option("display.max_columns", None)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sys
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

# Sube de notebooks/ a la raíz del repo

sys.path.append(str(Path.cwd().parent))

# Importamos funciones

import src.ml.functions as fc

%matplotlib inline 
%load_ext autoreload
%autoreload 2


## Cargamos dataset

In [ ]:
df = fc.load_data("dataset_modelo_proveedor_v1_synthetic.csv")

## Exploramos

In [ ]:
df.head()

In [ ]:
# Ordenamos por fecha

df["fecha_compra"] = pd.to_datetime(df["fecha_compra"], format="%Y-%m-%d", errors="coerce")
df = df.sort_values("fecha_compra")

### Shape, dtypes, datos perdidos

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

### Subset de entrenamiento sin datos perdidos

- Elegimos rellenar los datos perdidos numéricos e ignorar los que no.

In [ ]:
df_model = fc.df_model_knn(df)

## KNN

In [ ]:
X, y = fc.seleccionar_feature_cols(df_model)
X.shape, y.shape

### Train Test Split

- Temporal

In [ ]:
# Hago split temporal con función para enternar el modelo

X_train, X_test, y_train, y_test = fc.split_temporal(df_model)

In [ ]:
# Entrena KNN y genera predicciones explícitas (reproducible)

knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

knn.fit(X_train, y_train)
pred = knn.predict(X_test)
knn_acc = knn.score(X_test, y_test)
knn_acc


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

dummy = DummyClassifier(strategy="most_frequent", random_state=0).fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
labels_eval = np.union1d(y_test, pred)
labels_present_test = np.unique(y_test)

print(f"Dummy Accuracy (baseline de clase mayoritaria): {accuracy_score(y_test, dummy_pred):.4f}")
print(f"KNN Accuracy (acierto global): {accuracy_score(y_test, pred):.4f}")
print(f"KNN F1 Weighted (F1 ponderado por soporte de clase): {f1_score(y_test, pred, labels=labels_eval, average='weighted', zero_division=0):.4f}")
print(f"KNN Balanced Accuracy (media de recall por clase): {recall_score(y_test, pred, labels=labels_present_test, average='macro', zero_division=0):.4f}")


## Cierre Day 01 (Baseline)

### Dataset y split
- Dataset usado: `data/synthetic/dataset_modelo_proveedor_v1_synthetic.csv`.
- Este dataset contiene una fila por compra real (`fecha_compra` + proveedor elegido) y se usó para el baseline inicial de clasificación multiclase de proveedor.
- No contiene el universo completo de proveedores no elegidos por evento (eso se aborda en V2 candidates con otros modelos que lo soportan mejor).
- Split temporal: `train=13067`, `test=3267` (80/20 por orden temporal).

### Resultados
- Dummy Accuracy (baseline clase mayoritaria): `0.7677`
- KNN Accuracy (k=5): `0.5454`
- KNN F1 Weighted (k=5): `0.6140`
- KNN Balanced Accuracy (k=5): `0.1297`

### Diagnóstico
- Existe desbalance de clases y fuerte concentración histórica en proveedor dominante.
- KNN no supera al baseline `Dummy` en este corte temporal.
- Conclusión Day 01: baseline operativo sigue siendo `Dummy` como referencia inicial.
- Riesgo principal: optimizar solo `accuracy` en un problema desbalanceado; probablemente otros modelos darán mejor resultado que este.

## VISUALIZACIONES

In [ ]:

labels_eval = np.union1d(y_test, pred)
labels_present_test = np.unique(y_test)

metrics_df = pd.DataFrame([
    {
        "model": "Dummy",
        "accuracy": accuracy_score(y_test, dummy_pred),
        "f1_weighted": f1_score(y_test, dummy_pred, labels=labels_eval, average="weighted", zero_division=0),
        "balanced_accuracy": recall_score(y_test, dummy_pred, labels=labels_present_test, average="macro", zero_division=0),
    },
    {
        "model": "KNN (k=5)",
        "accuracy": accuracy_score(y_test, pred),
        "f1_weighted": f1_score(y_test, pred, labels=labels_eval, average="weighted", zero_division=0),
        "balanced_accuracy": recall_score(y_test, pred, labels=labels_present_test, average="macro", zero_division=0),
    },
])

metrics_df


In [ ]:
# BARPLOT COMPARACIÓN MODELOS

plot_df = metrics_df.melt(id_vars="model", var_name="metric", value_name="value")

plt.figure(figsize=(9, 4))
sns.barplot(data=plot_df, x="metric", y="value", hue="model")
plt.ylim(0, 1)
plt.title("Comparativa Dummy vs KNN")
plt.tight_layout()
plt.show()


In [ ]:
# DESBALANCE

train_dist = pd.Series(y_train).value_counts(normalize=True).head(10).rename("train_ratio")
test_dist = pd.Series(y_test).value_counts(normalize=True).head(10).rename("test_ratio")

dist_df = pd.concat([train_dist, test_dist], axis=1).fillna(0).reset_index().rename(columns={"index": "proveedor"})
dist_df
